# 02 — The protocol, and what it says

The first version of this comparison concluded that the semi-supervised arm improved recall
on the cancer class, 0.900 to 0.960. Three leaks stood behind that number, all of them
pushing the same way:

1. **31 of the 99 evaluation images were in the pre-training pool** (notebook 01);
2. the **clustering method** was chosen by ARI against every label, test folds included;
3. the **cluster-to-class alignment** was decided by a vote over those same labels.

And a confound: the semi-supervised arm took strictly more gradient steps than its baseline.

This notebook reads the artefacts of the corrected protocol. It is not there to defend a
conclusion — it is there to report one.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

## 1. What the corrected protocol does differently

* Duplicates are removed **by content**, and the evaluation set never loses an image.
* The clustering, the choice of method and the alignment happen **inside the training
  fold**. The test fold takes part in no decision.
* Three arms share folds, architecture and starting weights:

| arm | pre-training | what it isolates |
|---|---|---|
| `supervised` | none | the reference |
| `semi_supervised` | the fold's pseudo-labels | the supposed contribution |
| `permuted_control` | the same images, labels shuffled | budget and exposure |

* The checkpoint and the decision threshold come from an inner validation split carved out
  of the training fold.
* Five repeats of a five-fold cross-validation, so the spread is measured rather than
  guessed.

The third arm is the one that can end the discussion. It sees the same images and takes the
same steps; only the pairing between image and pseudo-label is destroyed.

In [ ]:
import json

from mri_semisupervised.config import EXPERIMENTS_DIR

corrected = EXPERIMENTS_DIR / "corrected"
print(f"reading {corrected.name}")

meta = json.loads((corrected / "manifest.json").read_text(encoding="utf-8"))
print(f"dataset fingerprint : {meta['dataset_fingerprint']}")
print(f"evaluation images   : {meta['evaluation_images']}")
print(f"unlabelled pool     : {meta['unlabelled_pool']}")
print(f"folds               : {meta['protocol']['n_splits']} x {meta['protocol']['n_repeats']} repeats")
print(f"gpu                 : {meta['versions']['gpu']}")

per_fold = pd.read_parquet(corrected / "per_fold.parquet")
predictions = pd.read_parquet(corrected / "predictions.parquet")
folds = pd.read_parquet(corrected / "folds.parquet")

## 2. The three arms, side by side

Two families of number, and they answer different questions. **ROC AUC and PR-AUC** say how
well the scores rank, whatever threshold is applied. **Recall and F1** say what happens at
the threshold that was actually chosen — on the inner validation, never on the test fold.

The original comparison reported only the second kind, which is how a model whose ranking
had got *worse* came to look better.

In [ ]:
headline = ["roc_auc", "pr_auc", "recall_positive", "f1_macro", "accuracy"]
per_fold.groupby("arm")[headline].agg(["mean", "std"]).round(3)

## 3. Paired comparisons

The arms share their folds, so the comparison is paired. Treating the two as independent
samples would throw the pairing away and widen every interval for nothing.

In [ ]:
from mri_semisupervised.protocol.uncertainty import paired_difference

pivot = per_fold.pivot(index="fold", columns="arm")
rows = []
for metric in ["roc_auc", "pr_auc", "recall_positive"]:
    for a, b in [
        ("semi_supervised", "supervised"),
        ("semi_supervised", "permuted_control"),
        ("permuted_control", "supervised"),
    ]:
        out = paired_difference(pivot[(metric, a)].to_numpy(), pivot[(metric, b)].to_numpy())
        rows.append({"metric": metric, "comparison": f"{a} - {b}", **out})
pd.DataFrame(rows).round(4)

The line to read first is `semi_supervised - permuted_control`. If its interval spans zero,
then pre-training on pseudo-labels does no better than pre-training on the same images with
those labels shuffled — and whatever the first version measured was budget and exposure, not
the information the clustering had found.

In [ ]:
from mri_semisupervised.viz.plots import plot_roc_compare

curves = {
    arm: (group["y_true"].to_numpy(), group["y_score"].to_numpy())
    for arm, group in predictions.groupby("arm")
}
plot_roc_compare(curves, title="Pooled out-of-fold ROC, five repeats")

## 4. Which clustering method each fold picked

Choosing the method inside the fold means the choice can differ from one fold to the next.
That is not instability to hide — it is a measurement of how much the choice depended on
seeing every label.

In [ ]:
print(folds["pseudo_method"].value_counts().to_string())
print()
print(f"ARI on the training labels: {folds['pseudo_ari_on_train'].mean():.3f} "
      f"+/- {folds['pseudo_ari_on_train'].std():.3f}")
print(f"pseudo-labels per fold    : {folds['n_pseudo'].mean():.0f}")

## 5. What the leaks were worth

The `legacy` run reproduces the original protocol faithfully — duplicates left in place,
method and alignment decided once over every label, checkpoint taken on training accuracy.
It exists so the difference is **measured** rather than quoted from an old file.

In [ ]:
legacy_dir = EXPERIMENTS_DIR / "legacy"
if (legacy_dir / "per_fold.parquet").exists():
    legacy = pd.read_parquet(legacy_dir / "per_fold.parquet")
    comparison = pd.concat(
        [
            per_fold.groupby("arm")[headline].mean().add_suffix("_corrected"),
            legacy.groupby("arm")[headline].mean().add_suffix("_legacy"),
        ],
        axis=1,
    )
    display(comparison.round(3))
else:
    print("no legacy run found: uv run python scripts/run_experiment.py --mode legacy")

## 6. What to take away

Whatever the numbers above say, the method is the point:

* an identity that comes from the data, so a duplicate cannot hide behind a folder;
* every decision that reads a label made inside the training fold;
* a control arm that makes "it helped" a falsifiable claim rather than a hopeful one;
* intervals, because with twenty images per fold a difference of one image moves recall by
  0.05.

A protocol that can only confirm what you hoped is not a protocol.